In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pkill -f streamlit
!pkill -f cloudflared

In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle

st.set_page_config(page_title="Wheat Yield Simulator", layout="wide")

st.title("🌾 Wheat Yield Scenario Simulator")
st.markdown("Select a location to load its historical climate baseline. Tweak the weather to predict the yield impact.")

# 1. Configuration & Asset Loading
st.sidebar.header("Configuration")
country = st.sidebar.radio("Select Country Model:", ["India", "USA"])

# File paths
paths = {
    "India": {
        "model": "/content/drive/MyDrive/wheat_yield_india/dataset/india_gb_model.pkl",
        "metadata": "/content/drive/MyDrive/wheat_yield_india/dataset/india_gb_metadata.pkl",
        "data": "/content/drive/MyDrive/wheat_yield_india/dataset/final_india_wheat_dataset_moderate_v2.csv"
    },
    "USA": {
        "model": "/content/drive/MyDrive/wheat_yield_india/dataset/usa_rf_model.pkl",
        "metadata": "/content/drive/MyDrive/wheat_yield_india/dataset/usa_rf_metadata.pkl",
        "data": "/content/drive/MyDrive/wheat_yield_india/dataset/final_usa_wheat_dataset_moderate.csv"
    }
}

@st.cache_resource
def load_assets(selected_country):
    with open(paths[selected_country]["model"], 'rb') as f:
        loaded_model = pickle.load(f)
    with open(paths[selected_country]["metadata"], 'rb') as f:
        loaded_meta = pickle.load(f)
    df = pd.read_csv(paths[selected_country]["data"])
    return loaded_model, loaded_meta, df

try:
    model, metadata, df = load_assets(country)
    expected_features = metadata['features']
    st.sidebar.success(f"{country} Assets Loaded!")
except Exception as e:
    st.error(f"Error loading files. Check your file paths!\n{e}")
    st.stop()

# US State Mapping Dictionary
us_states = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
}

# 2. Dynamic Cascading Geography Selection
st.subheader("1. Select Location")
col_loc1, col_loc2 = st.columns(2)

with col_loc1:
    states_list = sorted(df['State'].dropna().unique())

    # Use full names for USA, abbreviations otherwise
    if country == "USA":
        selected_state = st.selectbox(
            "State",
            states_list,
            format_func=lambda x: us_states.get(x, x)
        )
    else:
        selected_state = st.selectbox("State", states_list)

with col_loc2:
    districts_in_state = sorted(df[df['State'] == selected_state]['District'].dropna().unique())
    selected_district = st.selectbox("District", districts_in_state)

# Extract specific district data to calculate means
dist_data = df[(df['State'] == selected_state) & (df['District'] == selected_district)]
dist_code = dist_data['Dist Code'].iloc[0] if not dist_data.empty else None

# Calculate Default Means (Fallbacks to standard values if data is entirely missing)
mean_tmax = float(dist_data['Avg_Season_Tmax'].mean()) if not dist_data.empty and not pd.isna(dist_data['Avg_Season_Tmax'].mean()) else 30.0
mean_terminal = float(dist_data['Terminal_Heat_Tmax'].mean()) if not dist_data.empty and not pd.isna(dist_data['Terminal_Heat_Tmax'].mean()) else 35.0
mean_rain = float(dist_data['Total_Season_Rain'].mean()) if not dist_data.empty and not pd.isna(dist_data['Total_Season_Rain'].mean()) else 500.0
mean_sowing = float(dist_data['Sowing_Rainfall'].mean()) if not dist_data.empty and not pd.isna(dist_data['Sowing_Rainfall'].mean()) else 50.0

# 3. Interactive Weather Sliders
st.subheader(f"2. Simulate Climate Shocks for {selected_district}")
st.markdown("*Sliders default to the district's historical average. Adjust them to simulate weather anomalies.*")

col1, col2 = st.columns(2)

with col1:
    user_tmax = st.slider("Avg Season Tmax (°C)", min_value=10.0, max_value=50.0, value=mean_tmax, step=0.1)
    user_terminal = st.slider("Terminal Heat Tmax (°C)", min_value=10.0, max_value=55.0, value=mean_terminal, step=0.1)

with col2:
    user_rain = st.slider("Total Season Rain (mm)", min_value=0.0, max_value=2000.0, value=mean_rain, step=1.0)
    user_sowing = st.slider("Sowing Rainfall (mm)", min_value=0.0, max_value=500.0, value=mean_sowing, step=1.0)

# 4. Prediction Logic
st.markdown("---")
if st.button("🔮 Predict Yield Impact", type="primary", use_container_width=True):

    # Initialize empty input array with 0s matching exactly what the model expects
    input_df = pd.DataFrame(0, index=[0], columns=expected_features)

    # Fill Year and Historical Yield Stats (Invisible to user)
    if 'Year' in input_df.columns: input_df['Year'] = 2024
    if 'Yield_Lag2' in input_df.columns:
        input_df['Yield_Lag2'] = dist_data['Yield_Lag2'].mean() if not dist_data.empty else 2000.0
    if 'Yield_Rolling_Mean3' in input_df.columns:
        input_df['Yield_Rolling_Mean3'] = dist_data['Yield_Rolling_Mean3'].mean() if not dist_data.empty else 2000.0
    if 'Yield_Rolling_Std3' in input_df.columns:
        input_df['Yield_Rolling_Std3'] = dist_data['Yield_Rolling_Std3'].mean() if not dist_data.empty else 150.0

    # Fill Direct Slider Inputs
    if 'Avg_Season_Tmax' in input_df.columns: input_df['Avg_Season_Tmax'] = user_tmax
    if 'Terminal_Heat_Tmax' in input_df.columns: input_df['Terminal_Heat_Tmax'] = user_terminal
    if 'Total_Season_Rain' in input_df.columns: input_df['Total_Season_Rain'] = user_rain
    if 'Sowing_Rainfall' in input_df.columns: input_df['Sowing_Rainfall'] = user_sowing

    # Calculate Engineered Features on the fly!
    if 'Temperature_Anomaly' in input_df.columns:
        input_df['Temperature_Anomaly'] = user_tmax - mean_tmax
    if 'Rainfall_Deviation' in input_df.columns:
        input_df['Rainfall_Deviation'] = user_rain - mean_rain

    eps = 1e-6
    if 'Terminal_vs_AvgTemp' in input_df.columns:
        input_df['Terminal_vs_AvgTemp'] = user_terminal - user_tmax
    if 'SowingRain_to_TotalRain' in input_df.columns:
        input_df['SowingRain_to_TotalRain'] = user_sowing / (user_rain + 1.0)
    if 'Rain_per_Temp' in input_df.columns:
        input_df['Rain_per_Temp'] = user_rain / (user_tmax + eps)

    # Map the geographic One-Hot Encoded columns
    state_col = f"State_{selected_state}"
    district_col = f"District_{selected_district}"
    dist_code_col = f"Dist Code_{dist_code}"

    if state_col in input_df.columns: input_df[state_col] = 1
    if district_col in input_df.columns: input_df[district_col] = 1
    if dist_code_col in input_df.columns: input_df[dist_code_col] = 1

    # Execute Prediction
    try:
        prediction = model.predict(input_df)[0]

        # Determine appropriate full state name for success message
        display_state = us_states.get(selected_state, selected_state) if country == "USA" else selected_state

        st.success(f"Simulation Complete for {selected_district}, {display_state}")
        st.metric(label="Predicted Crop Yield", value=f"{prediction:,.2f} kg/hectare")

    except Exception as e:
        st.error(f"An error occurred: {e}")

Writing app.py


In [ ]:
!pip install -q streamlit
# Download cloudflared if you don't have it
!wget -q -O cloudflared-linux-amd64 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# Start Streamlit in the background
import subprocess
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Start the tunnel
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 88.3 MB/s eta 0:00:00
2026-03-13T03:26:39Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-13T03:26:39Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-13T03:26:42Z INF +--------------------------------------------------------------------------------------------+
2026-03-13T03:26:42Z INF |  Your quick Tunnel h